# Задачи

1. Выбрать два публичных API: основной (с пагинацией) и дополнительный (для обогащения данных).
1. Изучить контракт API: ресурсы, параметры, коды состояния, лимиты, версию.
1. Реализовать сбор ≥ 300 записей с корректной пагинацией, обработкой ошибок и ограничением скорости.
1. Нормализовать вложенный JSON в плоскую таблицу и выполнить обогащение из второго источника.
1. Оценить качество данных: полнота полей, дубликаты, типы, выбросы.
1. Реализовать инкрементальный повторный запуск (второй прогон не дублирует данные).
1. Воспроизвести упрощённый сценарий в Zapier или n8n и сравнить подходы количественно.
1. Оформить отчёт с обоснованием выбора инструментов и правовой оценкой сбора.

* Вариант 4
* Api - DummyJSON Posts + Comments (https://dummyjson.com/docs/posts)
* Доп. API - DummyJSON Users (https://dummyjson.com/docs/users)
* Целевая задача анализа - Активность авторов, связь постов, комментариев и реакций

In [ ]:
import json
import time
import random
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

import requests
import pandas as pd

OUT_DIR = Path("data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://dummyjson.com"


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    ),
    "Accept": "application/json",
}

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

def safe_get_json(
    url: str,
    params: Optional[dict] = None,
    timeout: int = 15,
) -> tuple[Optional[dict], int, str]:
    try:
        resp = session.get(url, params=params, timeout=timeout)
        content_type = resp.headers.get("Content-Type", "")
        
        if "application/json" not in content_type:
            return None, resp.status_code, f"Не JSON. Первые 200 байт: {resp.text[:200]}"
        if resp.status_code != 200:
            return None, resp.status_code, f"HTTP {resp.status_code}: {resp.text[:200]}"
        
        return resp.json(), resp.status_code, ""
    
    except requests.exceptions.Timeout:
        return None, 0, "Таймаут запроса"
    except requests.exceptions.RequestException as e:
        return None, 0, f"Ошибка запроса: {e}"

In [ ]:
class RateLimiter:
    def __init__(self, min_interval: float = 0.5):
        self.min_interval = min_interval
        self.last_call = 0.0

    def wait(self):
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call = time.time()

rate_limiter = RateLimiter(min_interval=0.6)  # 0.6 сек между запросами

def fetch_with_retry(
    url: str,
    params: Optional[dict] = None,
    max_retries: int = 3,
) -> tuple[Optional[dict], int, str]:
    """
    Выполняет запрос с повторами
    """
    for attempt in range(max_retries):
        rate_limiter.wait()
        data, status, err = safe_get_json(url, params)
        if status == 200 and data is not None:
            return data, status, ""

        if status == 429 or (500 <= status < 600):
            delay = (2 ** attempt) + random.uniform(0, 1)
            print(f"  Повтор через {delay:.1f} с (статус {status})")
            time.sleep(delay)
            continue

        return None, status, err

    return None, 0, f"Исчерпаны попытки ({max_retries})"

In [ ]:
def collect_paginated(
    endpoint: str,
    collection_key: str,
    page_size: int = 30,
    max_records: int = 500,
    max_pages: int = 20,
    extra_params: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """
    Собирает записи с пагинацией limit/skip.
    Возвращает (records, collection_log).
    """
    records = []
    log_entries = []
    skip = 0
    page = 1
    total_expected = None

    while len(records) < max_records and page <= max_pages:
        params = {"limit": page_size, "skip": skip}
        if extra_params:
            params.update(extra_params)

        url = f"{BASE_URL}{endpoint}"
        req_start = time.time()
        data, status, err = fetch_with_retry(url, params)
        duration = time.time() - req_start

        log_entry = {
            "page": page,
            "url": url,
            "params": json.dumps(params),
            "status": status,
            "records_on_page": 0,
            "duration_sec": round(duration, 3),
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

        if err or data is None:
            log_entry["error"] = err
            log_entries.append(log_entry)
            print(f"  Ошибка на странице {page}: {err}")
            break

        items = data.get(collection_key, [])
        if total_expected is None:
            total_expected = data.get("total", None)

        if not items:
            log_entry["note"] = "Пустая страница — остановка"
            log_entries.append(log_entry)
            break

        records.extend(items)
        log_entry["records_on_page"] = len(items)
        log_entries.append(log_entry)

        print(f"  Страница {page}: {len(items)} записей (всего {len(records)})")

        # Инкремент по фактической длине страницы
        skip += len(items)

        # Остановка по достижению total
        if total_expected is not None and skip >= total_expected:
            print(f"  Достигнут total={total_expected}")
            break

        page += 1

    return records, log_entries

In [7]:
print("Сбор постов...")
posts, posts_log = collect_paginated(
    endpoint="/posts",
    collection_key="posts",
    page_size=30,
    max_records=400,
    max_pages=15,
)
print(f"Собрано постов: {len(posts)}")

Сбор постов...
  Страница 1: 30 записей (всего 30)
  Страница 2: 30 записей (всего 60)
  Страница 3: 30 записей (всего 90)
  Страница 4: 30 записей (всего 120)
  Страница 5: 30 записей (всего 150)
  Страница 6: 30 записей (всего 180)
  Страница 7: 30 записей (всего 210)
  Страница 8: 30 записей (всего 240)
  Страница 9: 11 записей (всего 251)
  Достигнут total=251
Собрано постов: 251


In [8]:
print("Сбор комментариев...")
comments, comments_log = collect_paginated(
    endpoint="/comments",
    collection_key="comments",
    page_size=50,
    max_records=600,
    max_pages=15,
)
print(f"Собрано комментариев: {len(comments)}")

Сбор комментариев...
  Страница 1: 50 записей (всего 50)
  Страница 2: 50 записей (всего 100)
  Страница 3: 50 записей (всего 150)
  Страница 4: 50 записей (всего 200)
  Страница 5: 50 записей (всего 250)
  Страница 6: 50 записей (всего 300)
  Страница 7: 40 записей (всего 340)
  Достигнут total=340
Собрано комментариев: 340


In [9]:
print("Сбор пользователей...")
users, users_log = collect_paginated(
    endpoint="/users",
    collection_key="users",
    page_size=30,
    max_records=250,
    max_pages=10,
)
print(f"Собрано пользователей: {len(users)}")

Сбор пользователей...
  Страница 1: 30 записей (всего 30)
  Страница 2: 30 записей (всего 60)
  Страница 3: 30 записей (всего 90)
  Страница 4: 30 записей (всего 120)
  Страница 5: 30 записей (всего 150)
  Страница 6: 30 записей (всего 180)
  Страница 7: 28 записей (всего 208)
  Достигнут total=208
Собрано пользователей: 208


In [ ]:
def save_jsonl(records: list[dict], filepath: Path):
    with open(filepath, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(posts, OUT_DIR / "data_raw_posts.jsonl")
save_jsonl(comments, OUT_DIR / "data_raw_comments.jsonl")
save_jsonl(users, OUT_DIR / "data_raw_users.jsonl")


all_logs = posts_log + comments_log + users_log
log_df = pd.DataFrame(all_logs)
log_df.to_csv(OUT_DIR / "collection_log.csv", index=False)
print(f"Журнал сохранён: {len(log_df)} записей")

Журнал сохранён: 23 записей


In [ ]:
posts_df = pd.json_normalize(
    posts,
    sep="_",
    record_path=None,
    meta=None,
)
posts_df["tags_str"] = posts_df["tags"].apply(lambda x: "|".join(x) if isinstance(x, list) else "")

print("Столбцы после нормализации постов:", posts_df.columns.tolist())
posts_df.head(3)

Столбцы после нормализации постов: ['id', 'title', 'body', 'tags', 'views', 'userId', 'reactions_likes', 'reactions_dislikes', 'tags_str']


,id,title,body,tags,views,userId,reactions_likes,reactions_dislikes,tags_str
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,history|american|crime
1,2,He was an expert but not in a discipline,He was an expert but not in a discipline that ...,"[french, fiction, english]",4884,91,859,32,french|fiction|english
2,3,Dave watched as the forest burned up on the hill.,Dave watched as the forest burned up on the hi...,"[magical, history, french]",4152,16,1448,39,magical|history|french


In [ ]:
comments_df = pd.json_normalize(
    comments,
    sep="_",
    record_path=None,
    meta=None,
)

if "user" in comments_df.columns:
    user_normalized = pd.json_normalize(comments_df["user"], sep="_")
    user_normalized.columns = ["user_" + c for c in user_normalized.columns]
    comments_df = pd.concat([comments_df.drop(columns=["user"]), user_normalized], axis=1)

print("Столбцы после нормализации комментариев:", comments_df.columns.tolist())
comments_df.head(3)

Столбцы после нормализации комментариев: ['id', 'body', 'postId', 'likes', 'user_id', 'user_username', 'user_fullName']


,id,body,postId,likes,user_id,user_username,user_fullName
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson


In [ ]:
users_df = pd.json_normalize(
    users,
    sep="_",
    record_path=None,
    meta=None,
)

print("Столбцы после нормализации пользователей:", users_df.columns.tolist())
users_df.head(3)

Столбцы после нормализации пользователей: ['id', 'firstName', 'lastName', 'maidenName', 'age', 'gender', 'email', 'phone', 'username', 'password', 'birthDate', 'image', 'bloodGroup', 'height', 'weight', 'eyeColor', 'ip', 'macAddress', 'university', 'ein', 'ssn', 'userAgent', 'role', 'hair_color', 'hair_type', 'address_address', 'address_city', 'address_state', 'address_stateCode', 'address_postalCode', 'address_coordinates_lat', 'address_coordinates_lng', 'address_country', 'bank_cardExpire', 'bank_cardNumber', 'bank_cardType', 'bank_currency', 'bank_iban', 'company_department', 'company_name', 'company_title', 'company_address_address', 'company_address_city', 'company_address_state', 'company_address_stateCode', 'company_address_postalCode', 'company_address_coordinates_lat', 'company_address_coordinates_lng', 'company_address_country', 'crypto_coin', 'crypto_wallet', 'crypto_network']


,id,firstName,lastName,maidenName,age,gender,email,phone,username,password,...,company_address_city,company_address_state,company_address_stateCode,company_address_postalCode,company_address_coordinates_lat,company_address_coordinates_lng,company_address_country,crypto_coin,crypto_wallet,crypto_network
0,1,Emily,Johnson,Smith,29,female,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,emilyspass,...,San Francisco,Wisconsin,WI,37657,71.814525,-161.150263,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)
1,2,Michael,Williams,,36,male,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,michaelwpass,...,Los Angeles,New Hampshire,NH,73442,79.098326,-119.624845,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)
2,3,Sophia,Brown,,43,female,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,sophiabpass,...,Dallas,Nevada,NV,88511,20.086743,-34.577107,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)


In [15]:
# Посты содержат userId. Соединяем с users по id
users_slim = users_df[["id", "firstName", "lastName", "username", "email", "company_name", "address_city", "address_country"]].copy()
users_slim = users_slim.rename(columns={"id": "userId"})

before = len(posts_df)
enriched_df = posts_df.merge(users_slim, on="userId", how="left")
after = len(enriched_df)

print(f"Строк до merge: {before}, после merge: {after}")
assert before == after, "Merge изменил число строк!"

# Добавим количество комментариев к каждому посту
comment_counts = comments_df.groupby("postId").size().reset_index(name="comment_count")
enriched_df = enriched_df.merge(comment_counts, left_on="id", right_on="postId", how="left")
enriched_df["comment_count"] = enriched_df["comment_count"].fillna(0).astype(int)

print(f"Итоговая таблица: {enriched_df.shape}")
enriched_df.head(3)

Строк до merge: 251, после merge: 251
Итоговая таблица: (251, 18)


,id,title,body,tags,views,userId,reactions_likes,reactions_dislikes,tags_str,firstName,lastName,username,email,company_name,address_city,address_country,postId,comment_count
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25,history|american|crime,Ava,Harris,avahx,ava.harris@x.dummyjson.com,Lebsack Inc,San Antonio,United States,1.0,3
1,2,He was an expert but not in a discipline,He was an expert but not in a discipline that ...,"[french, fiction, english]",4884,91,859,32,french|fiction|english,Levi,Hicks,levih,levi.hicks@x.dummyjson.com,Quitzon LLC,Phoenix,United States,2.0,2
2,3,Dave watched as the forest burned up on the hill.,Dave watched as the forest burned up on the hi...,"[magical, history, french]",4152,16,1448,39,magical|history|french,Avery,Perez,averyp,avery.perez@x.dummyjson.com,Herzog Inc,Columbus,United States,3.0,1


In [ ]:
def completeness_table(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    comp = df.notna().sum() / total * 100
    result = comp.reset_index()
    result.columns = ["column", "completeness_%"]
    result["completeness_%"] = result["completeness_%"].round(1)
    return result.sort_values("completeness_%", ascending=False)

print("Полнота полей (итоговая таблица):")
completeness_table(enriched_df).head(20)

Полнота полей (итоговая таблица):


,column,completeness_%
0,id,100.0
1,title,100.0
2,body,100.0
3,tags,100.0
4,views,100.0
5,userId,100.0
6,reactions_likes,100.0
7,reactions_dislikes,100.0
8,tags_str,100.0
9,firstName,100.0


In [ ]:
dup_posts = enriched_df.duplicated(subset=["id"]).sum()
dup_comments = comments_df.duplicated(subset=["id"]).sum()
dup_users = users_df.duplicated(subset=["id"]).sum()
print(f"Дубликаты по id: posts={dup_posts}, comments={dup_comments}, users={dup_users}")

Дубликаты по id: posts=0, comments=0, users=0


In [ ]:
enriched_df["id"] = pd.to_numeric(enriched_df["id"], errors="coerce").astype("Int64")
enriched_df["userId"] = pd.to_numeric(enriched_df["userId"], errors="coerce").astype("Int64")
enriched_df["views"] = pd.to_numeric(enriched_df["views"], errors="coerce").astype("Int64")
enriched_df["reactions_likes"] = pd.to_numeric(enriched_df["reactions_likes"], errors="coerce").astype("Int64")
enriched_df["reactions_dislikes"] = pd.to_numeric(enriched_df["reactions_dislikes"], errors="coerce").astype("Int64")
enriched_df["comment_count"] = pd.to_numeric(enriched_df["comment_count"], errors="coerce").astype("Int64")

errors = []
if (enriched_df["views"] < 0).any():
    errors.append("views < 0")
if (enriched_df["reactions_likes"] < 0).any():
    errors.append("likes < 0")
if (enriched_df["comment_count"] < 0).any():
    errors.append("comment_count < 0")
if errors:
    print("Ошибки:", errors)
else:
    print("Все проверки пройдены")

Все проверки пройдены


In [ ]:
STATE_FILE = OUT_DIR / "state.json"

def load_state() -> dict:
    if STATE_FILE.exists():
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_state(state: dict):
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()
last_post_id = state.get("last_post_id", 0)

# Фильтруем только новые посты (id > last_post_id)
new_posts_df = enriched_df[enriched_df["id"] > last_post_id].copy()
print(f"Новых постов: {len(new_posts_df)} (водяной знак: {last_post_id})")

if len(new_posts_df) > 0:
    state["last_post_id"] = int(new_posts_df["id"].max())
    save_state(state)
    print(f"Последний id обновлён: {state['last_post_id']}")
else:
    print("Новых записей нет")

Новых постов: 0 (водяной знак: 251)
Новых записей нет — повторный запуск не создал дубликатов


In [23]:
enriched_df.to_csv(OUT_DIR / "data_clean.csv", index=False)
print(f"Сохранено: {OUT_DIR / 'data_clean.csv'} ({len(enriched_df)} строк)")

Сохранено: data\data_clean.csv (251 строк)
